# Anchored PC: Stage A diagnostic
Fixed source, BERT-Mini checkpoint and news cache. No training loop.
Use a free T4 GPU if available. The diagnostic stops at a failed anchor, failed convergence, or a two-hour budget. No Drive mount or GitHub push is needed.
Source base: `68088cb22fe80e258d760ea5690aa1e9603383f4`. Bundle SHA-256: `85923dab55dce8de03c5b4abc5d96275f88756f7033b24fb1bce774f0a0ffd65`.


In [ ]:
from google.colab import files
from pathlib import Path
bundle = Path('/content/anchored_pc_bundle.tar.gz')
if not bundle.exists():
    uploaded = files.upload()
    assert bundle.exists(), 'Upload anchored_pc_bundle.tar.gz'


In [ ]:
import hashlib, json, tarfile, subprocess, sys, os
assert hashlib.sha256(bundle.read_bytes()).hexdigest() == '85923dab55dce8de03c5b4abc5d96275f88756f7033b24fb1bce774f0a0ffd65'
root = Path('/content/anchored_pc')
root.mkdir(exist_ok=True)
with tarfile.open(bundle) as archive:
    archive.extractall(root, filter='data')
manifest = json.loads((root / 'bundle_manifest.json').read_text())
for name, expected in manifest['files'].items():
    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == expected, name
print('Verified', len(manifest['files']), 'source and input files')
(root / 'src/tests/__init__.py').touch(exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'jax-cuda13-plugin', 'jax-cuda13-pjrt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'jax[cuda12]==0.11.2', 'optax==0.2.8', 'flax==0.12.10',
    'numpy==2.5.4', 'tokenizers==0.23.3', 'pytest', 'hypothesis',
    'matplotlib', 'datasets', 'optuna', 'tqdm', 'jaxtyping'], check=True)
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['PYTHONPATH'] = str(root / 'src')
os.chdir(root / 'src')
subprocess.run([sys.executable, '-c', 'import jax; print(jax.__version__, jax.devices())'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(root / 'src')], check=True)


In [ ]:
# Unit and tiny-model checks run on CPU, independent of the GPU diagnostic.
test_env = dict(os.environ, JAX_PLATFORMS='cpu')
checked = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_anchored_pc.py',
    '-q', '-p', 'no:cacheprovider'], env=test_env, capture_output=True, text=True, timeout=300)
print(checked.stdout, checked.stderr)
checked.check_returncode()
probe = subprocess.run([sys.executable, '-c', 'import jax; print(jax.__version__, jax.devices())'], capture_output=True, text=True, check=True)
print(probe.stdout, probe.stderr)


In [ ]:
# Stage A only. Incremental evidence survives a failed scientific gate.
import datetime, threading
run_dir = root / ('results_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
command = [sys.executable, '-u', '-m', 'fabricpc.experiments.anchored_pc',
    '--checkpoint', str(root / 'checkpoints/bert_mini_128.json'),
    '--data-dir', str(root / 'data/domain_forgetting'),
    '--output-dir', str(run_dir), '--budget-seconds', '7140']
(root / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
watchdog = threading.Timer(7200, process.kill)
watchdog.daemon = True
watchdog.start()
try:
    with (root / 'process.log').open('w') as log:
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
    print('Process exit code:', process.wait())
finally:
    watchdog.cancel()
summary = json.loads((run_dir / 'summary.json').read_text())
print(json.dumps({k: summary.get(k) for k in ['status','decision','reason','elapsed_seconds','selected_solver']}, indent=2))


In [ ]:
# Export complete or partial evidence, source hashes and environment.
import shutil
shutil.copy(root / 'environment.txt', run_dir / 'environment.txt')
shutil.copy(root / 'bundle_manifest.json', run_dir / 'bundle_manifest.json')
shutil.copy(root / 'process.log', run_dir / 'process.log')
(run_dir / 'colab_tests.log').write_text(checked.stdout + checked.stderr)
hardware = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
(run_dir / 'hardware.txt').write_text(hardware.stdout + hardware.stderr)
evidence = shutil.make_archive(str(run_dir), 'zip', run_dir)
files.download(evidence)
